---
title: Example 1. Consumption function in code
course: Example course (pymetrix)
links: [least-squares, methods/OLS]
---

# Example 1. Consumption function in code

The same estimation as Lecture 1, in code. The data come with pymetrix, so the notebook runs from any folder.

In [ ]:
from pymetrix import Dataset, Equation, OLS

data = Dataset.example("sample_macro")
fit = Equation("c ~ y + c[-1]").fit(data, method=OLS(cov="hac"))
fit.summary()

Check the residuals for serial correlation:

In [ ]:
fit.test("breusch_godfrey", lags=4)

## Simultaneity

Income includes consumption, so OLS credits income with part of every consumption shock. The data
were built with 0.6 on income and 0.2 on lagged consumption. Instruments that move income but not
the consumption shock (government spending, and income last quarter) recover them:

In [ ]:
from pymetrix import IV

iv = Equation("c ~ y + c[-1]").fit(data, method=IV(endog=["y"], instruments=["g", "y[-1]"]))
compare = {"OLS": fit.params.values.round(3), "IV": iv.params.values.round(3), "true": [None, 0.6, 0.2]}
assert abs(iv.params["y"] - 0.6) < 0.05 and fit.params["y"] > iv.params["y"] + 0.1  # the point of the exercise
compare